# 배포 후보를 시험하고 이전 버전으로 복구하기

서버가 켜진 것과 새 버전이 기존 데이터를 읽는 것은 서로 다른 검사다. 변경 전후에 같은 서비스 요청을 보낸다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 어디까지 성공해야 배포가 끝날까?

이미지 빌드가 성공해도 서버에서 필요한 환경 변수가 비어 있을 수 있다. 첫 요청이 성공한 뒤에도 과거 버전이 만든 행만 읽지 못할 수 있다.

그래서 버전 응답, 기존 ID 읽기, 새 ID 쓰고 다시 읽기 같은 검사를 분리한다. 실패 시 어느 단계까지 통과했는지가 남아야 복구 위치를 고를 수 있다.

아래에서는 배포 후보 판정 함수를 로컬에서 시험한다. 실제 EC2 인스턴스와 디스크는 수업 환경에 배포한 뒤 같은 입력으로 확인한다.

## 200을 돌려주는 실패한 배포

버전은 맞지만 배포 전에 저장한 T1이 사라진 상태다. 검사 결과를 하나의 True로 뭉개지 않고 처음 실패한 경계를 남긴다.

In [1]:
def failed_checks(report):
    checks = {"상태 코드": report["http"] == 200,
              "앱 버전": report["version"] == "v2",
              "기존 접수": report["existing"] == "T1"}
    return [name for name, passed in checks.items() if not passed]

probe = {"http":200, "version":"v2", "existing":None}
print(failed_checks(probe))

['기존 접수']


상태 페이지는 정상이지만 기존 접수 조회가 실패했다. 점검 전에 빈 DB나 새 볼륨에서 서비스를 열었다면 이런 결과가 나온다.

### 이전 버전으로 잘못 연결된 후보 잡기

기존 접수가 보이지만 버전은 v1인 응답을 만든다. 어떤 경계를 먼저 조사해야 할지 출력과 대조한다.

In [2]:
old = dict(probe, existing="T1", version="v1")

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>print(failed_checks(old))</code></pre><pre>[&#x27;앱 버전&#x27;]
</pre><p>볼륨 복구보다 먼저 배포 대상 태그와 트래픽이 향한 서비스 버전을 확인한다.</p></details>

## 저장소가 바뀌는 실패를 파일로 재현하기

상태 응답을 꾸미는 데서 멈추지 않고 두 개의 실제 SQLite 파일을 열어 잘못된 볼륨을 선택한 상황을 재현한다. EC2나 컨테이너 없이 저장소 경로의 효과만 관찰한다.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sqlite3
storage = TemporaryDirectory()
root = Path(storage.name)
def opened_ids(path):
    db = sqlite3.connect(path)
    try:
        db.execute("CREATE TABLE IF NOT EXISTS ticket(id TEXT PRIMARY KEY)")
        db.commit()
        return [r[0] for r in db.execute("SELECT id FROM ticket ORDER BY id")]
    finally:
        db.close()
original = root/"existing.db"
opened_ids(original)
with sqlite3.connect(original) as seed:
    seed.execute("INSERT INTO ticket VALUES ('T1')")
print(opened_ids(root/"new.db"))

[]


서비스는 새 파일을 열고 오류 없이 빈 테이블까지 만들었다. 프로세스가 살아 있는지만 확인하면 기존 접수 누락을 발견하지 못한다.

### 디스크를 다시 연결한 뒤 읽기 검사하기

배포 후보가 원래 경로를 가리키도록 복원한다. 같은 ID가 다시 보이는지 읽고 임시 DB에도 데이터가 복제된 것은 아닌지 비교한다.

In [5]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>print(opened_ids(original), opened_ids(root/&quot;new.db&quot;))</code></pre><pre>[&#x27;T1&#x27;] []
</pre><p>원래 DB에서 T1을 읽었고 새 파일은 비어 있다. 경로를 고치는 것과 데이터를 복원하는 것은 다른 작업이다.</p></details>

## 다른 입력에 적용하기

GitHub Actions에 팀의 기존 ID 읽기 검사를 연결한다. 시험 환경에서 이전 이미지를 다시 띄운 뒤 기존 데이터가 보이는지 확인하고, 새 스키마가 이전 코드와 호환되는 경우와 그렇지 않은 경우를 구분한다.

실패 로그, 배포 커밋, 이미지 태그, 복구 후 조회 결과를 하나의 인계 메모에 남긴다.